In [2]:
import geopandas as gpd
import pandas as pd

# 1. Wczytanie siatki GUS
print("Wczytywanie siatki GUS (GeoJSON)...")
grid_gus = gpd.read_file("../data/1km_grid.geojson")
if grid_gus.crs != "EPSG:2180":
    grid_gus = grid_gus.to_crs(epsg=2180)

# 2. Wczytanie granic powiatow (ten sam plik, ktorego uzywa mapa do choropleth)
print("Wczytywanie granic powiatow...")
granice_powiatow = gpd.read_file("../data/powiaty_teryt.geojson")
if granice_powiatow.crs != "EPSG:2180":
    granice_powiatow = granice_powiatow.to_crs(epsg=2180)

# ---------- KROK KLUCZOWY: przypisz KAZDA komorke siatki do powiatu ----------
print("Przypisywanie CALEJ siatki do powiatow (to moze chwile potrwac)...")
grid_gus["geometry_centroid"] = grid_gus.geometry.centroid
grid_centroidy = grid_gus.set_geometry("geometry_centroid")

grid_z_powiatem = gpd.sjoin(
    grid_centroidy[["geometry_centroid", "tot"]],
    granice_powiatow[["geometry", "JPT_KOD_JE"]],
    how="left",
    predicate="within"
)
grid_z_powiatem["JPT_KOD_JE"] = grid_z_powiatem["JPT_KOD_JE"].astype(str).str.zfill(4)

# ---------- WAZNE: 45 obszarow metropolitalnych to POLACZENIE dwoch prawdziwych
# powiatow (np. "Krakow + krakowski" = miasto Krakow (teryt 1261) + powiat
# krakowski (teryt 1206)), bo CEPiK nie pozwalal ich rozdzielic przy liczeniu
# floty EV. Kazda lokalizacja zachowuje swoj PRAWDZIWY, osobny kod TERYT.
df_mapowanie = pd.read_csv("../data/candidate_locations_rozszerzone.csv", low_memory=False)
# UWAGA: samo "powiat_nazwa" NIE WYSTARCZY jako klucz - w Polsce jest 10
# nazw powiatow powtarzajacych sie w roznych wojewodztwach (np. powiat
# nowodworski istnieje i na Pomorzu, i na Mazowszu - to dwa rozne miejsca).
# Ten sam problem juz raz rozwiazywalismy wczesniej w projekcie (integracja
# danych NSP) - uzywajac zlozonego klucza wojewodztwo+nazwa.
df_mapowanie["_klucz_powiatu"] = (
    df_mapowanie["powiat_wojewodztwo"].astype(str) + "|" + df_mapowanie["powiat_nazwa"].astype(str)
)
mapowanie_teryt_na_nazwe = (
    df_mapowanie.assign(teryt_str=df_mapowanie["teryt_powiat_geo"].astype(str).str.zfill(4))
    .drop_duplicates(subset="teryt_str")
    .set_index("teryt_str")["_klucz_powiatu"]
)
grid_z_powiatem["powiat_nazwa_polaczona"] = grid_z_powiatem["JPT_KOD_JE"].map(mapowanie_teryt_na_nazwe)

suma_siatki_per_powiat = grid_z_powiatem.groupby("powiat_nazwa_polaczona")["tot"].sum()
print(f"Przypisano {len(grid_z_powiatem)} komorek do {suma_siatki_per_powiat.index.nunique()} obszarow (powiatow/polaczonych metropolii).")

# ---------- Walidacja: porownanie z oficjalna populacja GUS (powiat_ludnosc) ----------
kontrola = df_mapowanie.drop_duplicates(subset="_klucz_powiatu")[["_klucz_powiatu","powiat_nazwa","powiat_ludnosc"]].copy()
kontrola["suma_z_siatki"] = kontrola["_klucz_powiatu"].map(suma_siatki_per_powiat)
kontrola["roznica_proc"] = (kontrola["suma_z_siatki"] / kontrola["powiat_ludnosc"] - 1) * 100

print("\n=== Kontrola jakosci: siatka GUS vs oficjalna populacja (powiat_ludnosc) ===")
print(f"Brak dopasowania (NaN): {kontrola['suma_z_siatki'].isna().sum()} z {len(kontrola)} powiatow")
print(f"Srednia roznica: {kontrola['roznica_proc'].mean():+.1f}%")
print(f"Mediana roznicy: {kontrola['roznica_proc'].median():+.1f}%")
print(f"Zakres: {kontrola['roznica_proc'].min():+.1f}% do {kontrola['roznica_proc'].max():+.1f}%")
print("\nNajwieksze rozbieznosci (5 powiatow):")
print(kontrola.reindex(kontrola["roznica_proc"].abs().sort_values(ascending=False).index)
      [["powiat_nazwa","powiat_ludnosc","suma_z_siatki","roznica_proc"]].head(5).to_string(index=False))

# 3. Wczytanie kandydatow i dopasowanie ich WLASNEJ komorki
print("\nWczytywanie lokalizacji kandydatow...")
df = df_mapowanie.copy()
kolumny_do_usuniecia = ["index_right", "tot", "udzial_populacji", "_teryt_4cyfry", "_suma_siatki_powiat"]
df = df.drop(columns=[k for k in kolumny_do_usuniecia if k in df.columns])
candidates_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
).to_crs(epsg=2180)

candidates_with_grid = gpd.sjoin(
    candidates_gdf,
    grid_gus[["geometry", "tot"]],
    how="left",
    predicate="within"
)
candidates_with_grid["tot"] = candidates_with_grid["tot"].fillna(0)

# ---------- Finalny udzial: wlasna komorka / SUMA CALEJ SIATKI W POLACZONYM OBSZARZE ----------
candidates_with_grid["_suma_siatki_powiat"] = candidates_with_grid["_klucz_powiatu"].map(
    suma_siatki_per_powiat
).fillna(1)

liczba_niedopasowanych = (candidates_with_grid["_suma_siatki_powiat"] == 1).sum()
if liczba_niedopasowanych > 100:
    print(f"UWAGA: {liczba_niedopasowanych} wierszy nie dopasowalo powiatu - "
          f"sprawdz format nazw powiatow w obu plikach recznie!")

candidates_with_grid["udzial_populacji"] = (
    candidates_with_grid["tot"] / candidates_with_grid["_suma_siatki_powiat"]
)

liczba_kandydatow_w_komorce = candidates_with_grid.groupby(
    ["_klucz_powiatu", "index_right"]
)["location_id"].transform("count")
candidates_with_grid["udzial_populacji"] = (
    candidates_with_grid["udzial_populacji"] / liczba_kandydatow_w_komorce
)

# ---------- weryfikacja koncowa ----------
suma_kontrolna = candidates_with_grid.groupby("_klucz_powiatu")["udzial_populacji"].sum()
print(f"\nKontrola: suma udzial_populacji per powiat - min={suma_kontrolna.min():.6f}, max={suma_kontrolna.max():.6f}")
print(f"(powinno byc <=1.0 - populacja przypisana kandydatom nie moze przekraczac calej populacji z siatki dla tego powiatu)")

output_path = "../data/candidate_locations_po_korekcie_gus.csv"
candidates_with_grid.drop(columns="geometry").to_csv(output_path, index=False)
print(f"\nSukces! Zapisano plik: {output_path}")

Wczytywanie siatki GUS (GeoJSON)...
Wczytywanie granic powiatow...
Przypisywanie CALEJ siatki do powiatow (to moze chwile potrwac)...
Przypisano 315857 komorek do 335 obszarow (powiatow/polaczonych metropolii).

=== Kontrola jakosci: siatka GUS vs oficjalna populacja (powiat_ludnosc) ===
Brak dopasowania (NaN): 0 z 335 powiatow
Srednia roznica: +2.2%
Mediana roznicy: +2.5%
Zakres: -8.7% do +18.3%

Najwieksze rozbieznosci (5 powiatow):
  powiat_nazwa  powiat_ludnosc  suma_z_siatki  roznica_proc
Świętochłowice           45021          53244     18.264810
       Chorzów           99420         112191     12.845504
     gryfiński           76489          83691      9.415733
       gdański          133490         121822     -8.740730
         Sopot           31444          29140     -7.327312

Wczytywanie lokalizacji kandydatow...

Kontrola: suma udzial_populacji per powiat - min=0.115085, max=1.098989
(powinno byc <=1.0 - populacja przypisana kandydatom nie moze przekraczac calej populacji

In [13]:
import geopandas as gpd

# Wczytanie siatki (pamiętaj o ścieżce poziom wyżej)
grid_gus = gpd.read_file("../data/1km_grid.geojson")

# Wyświetlenie wszystkich nazw kolumn w pliku
print("Dostępne kolumny w siatce GUS:")
print(grid_gus.columns.tolist())

Dostępne kolumny w siatce GUS:
['oid', 'shape_leng', 'shape_area', 'code', 'pl_code', 'tot', 'tot_male', 'tot_fem', 'tot_0_14', 'tot_15_64', 'tot_65_', 'multipolygon_area', 'multipolygon_length', 'geometry']
